# 02. Treinar Splink (link_only Censo × CPF)

Profile, blocking pré-treino, treino do modelo, predict e clustering.

Usa `link_type='link_only'`: só gera pares **entre** Censo e CPF.
A coorte não entra aqui. Validação: labels em
[`03_validar_coorte.ipynb`](03_validar_coorte.ipynb); lista ouro em
[`04_validar_lista_ouro.ipynb`](04_validar_lista_ouro.ipynb).

**1ª passada:** sem nome da mãe. Nomes fonéticos com exact + JW 0,95 e 0,90.
`data_nascimento` é ExactMatch (string ISO). Idade exact e ±1. UF no score.
Blocking por partes da DOB (ano; mês+dia; mês+ano) e DOB completa + um lado do nome.

Requer Splink 5 (`pip install 'splink==5.0.0.dev1'`).

**EDA descritiva:** [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).


In [25]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_LIMPA,
    USE_PHONETIC_STRIP_VOWELS,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra profile/blocking: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}')


OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: 21
FILTRO_MUNICIPIO: None
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2023
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
Removidas 22 tabelas/views residuais __splink__*
splink_input → registro_limpo
Registros: 14,616,377
DuckDB: threads=20, memory_limit=279.3 GiB (defaults: 20, 300GB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Amostra profile/blocking: 1,000,000 de 14,616,377


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.

In [26]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
    'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


,origem,n
0,cpf,8026573
1,censo,6589804


,0,1
origem,censo,cpf
pct_primeiro_nome,100.0,100.0
pct_ultimo_nome,99.6,100.0
pct_nome_completo,100.0,100.0
pct_nome_meio,88.4,96.2
pct_nome_mae,33.8,96.8
pct_primeiro_nome_mae,33.8,96.8
pct_nome_meio_mae,30.0,90.8
pct_ultimo_nome_mae,33.8,96.8
pct_data_nascimento,85.1,96.9


## Exploração pré-modelo

Profile Splink das colunas de linkage e análise de blocking (cumulativo + maiores blocos).

In [27]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'sexo', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'sexo', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'sexo', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'sexo', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'sexo', 'data_nascimento'),    
    block_on('primeiro_nome_phon', 'sexo', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'sexo', 'mes_nascimento', 'ano_nascimento'),        
    block_on('ultimo_nome_phon', 'sexo', 'mes_nascimento', 'dia_nascimento'),
    block_on('ultimo_nome_phon', 'sexo', 'mes_nascimento', 'ano_nascimento'),
]

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome_phon', 'ultimo_nome_phon', 'nome_completo_phon',
        'uf', 'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
        'idade',
    ],
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.VConcatChart(...)

In [28]:
# Análise de blocking só entre origens (link_only), na amostra ou na base completa.
con.execute(f'''
CREATE OR REPLACE VIEW splink_analysis_censo AS
SELECT * FROM {analysis_table} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_analysis_cpf AS
SELECT * FROM {analysis_table} WHERE origem = 'cpf'
''')

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=['splink_analysis_censo', 'splink_analysis_cpf'],
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='link_only',
)


alt.Chart(...)

## Modelo Splink

Settings e `Linker` em `link_only`. Comparisons: nomes fonéticos (JW 0,95 e 0,90),
ExactMatch em `data_nascimento`, idade (exact e ±1), sexo, UF. Sem `nome_mae*`
e sem CEP. UF no score, não no blocking.


In [29]:
from splink import Linker, SettingsCreator
import splink.comparison_library as cl
import splink.comparison_level_library as cll

input_cols = set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)

comparisons = [
    cl.NameComparison('nome_completo_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.NameComparison('primeiro_nome_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.NameComparison('nome_meio_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.NameComparison('ultimo_nome_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.ExactMatch('data_nascimento').configure(term_frequency_adjustments=True),
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('idade'),
            cll.ExactMatchLevel('idade'),
            cll.AbsoluteDifferenceLevel('idade', 1),
            cll.ElseLevel(),
        ],
    ),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    cl.ExactMatch('uf').configure(term_frequency_adjustments=True),
]

if USE_PHONETIC_STRIP_VOWELS and 'nome_completo_phon_sv' in input_cols:
    comparisons.append(cl.NameComparison('nome_completo_phon_sv'))

con.execute(f"""
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
""")
con.execute(f"""
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
""")
print(
    'splink_censo:', con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0],
    '| splink_cpf:', con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0],
)

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
)
linker = Linker(
    ['splink_censo', 'splink_cpf'],
    settings,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


splink_censo: 6589804 | splink_cpf: 8026573


In [30]:
deterministic_rules = [
    block_on('nome_completo_phon', 'data_nascimento'),    
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=10_000_000)
# EM em sexo+DOB (sem CEP): observa discordância de nome.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('sexo', 'data_nascimento'),
    estimate_without_term_frequencies=True,
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Probability two random records match is estimated to be  8.07e-08.
This means that amongst all possible pairwise record comparisons, one in 12,389,613.41 are expected to match.  With 52,893,542,861,692 total possible comparisons, we expect a total of around 4,269,184.29 matching pairs
----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

u probability not trained for uf - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the training data.

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nome_completo_phon (no m values are trained).
    - primeiro_nome_phon (no m values are trained).
    - nome_meio_phon (no m values are trained).
    - ultimo_nome_phon (no m values are trained).
    - data_nascimento (no m values are trained).
    - idade (no m values are trained).
    - sexo (no m values are trained).
    - uf (some u values are not trained, no m values are trained).


...


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."sexo" = r."sexo") AND (l."data_nascimento" = r."data_nascimento")

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - primeiro_nome_phon
    - nome_meio_phon
    - ultimo_nome_phon
    - idade
    - uf

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - data_nascimento
    - sexo


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Level All other comparisons on comparison idade not observed in dataset, unable to train m value

Level All other comparisons on comparison uf not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.296 in the m_probability of nome_completo_phon, level `Exact match on nome_completo_phon`
Iteration 2: Largest change in params was 0.314 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 3: Largest change in params was 0.454 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 4: Largest change in params was 0.593 in the m_probability of primeiro_nome_phon, level `All other comparisons`
Iteration 5: Largest change in params was 0.628 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.0288 in probability_two_random_records_match
Iteration 7: Largest change in params was 0.00157 in probability_two_random_records_match
Iteration 8: Largest change in params 

<EMTrainingSession, blocking on (l."sexo" = r."sexo") AND (l."data_nascimento" = r."data_nascimento"), deactivating comparisons data_nascimento, sexo>

In [31]:
# EM em primeiro_nome_phon + DOB: observa m de sobrenome, completo, idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primeiro_nome_phon" = r."primeiro_nome_phon") AND (l."data_nascimento" = r."data_nascimento")

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - nome_meio_phon
    - ultimo_nome_phon
    - idade
    - sexo
    - uf

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - primeiro_nome_phon
    - data_nascimento


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Level All other comparisons on comparison idade not observed in dataset, unable to train m value

Level All other comparisons on comparison uf not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 0.52 in probability_two_random_records_match
Iteration 2: Largest change in params was 0.456 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0169 in probability_two_random_records_match
Iteration 4: Largest change in params was 0.00139 in probability_two_random_records_match
Iteration 5: Largest change in params was 6.63e-05 in probability_two_random_records_match

EM converged after 5 iterations
m probability not trained for idade - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the training data.
m probability not trained for uf - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the trai

<EMTrainingSession, blocking on (l."primeiro_nome_phon" = r."primeiro_nome_phon") AND (l."data_nascimento" = r."data_nascimento"), deactivating comparisons primeiro_nome_phon, data_nascimento>

In [32]:
# EM em primeiro_nome_phon + DOB: observa m de sobrenome, completo, idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'ultimo_nome_phon','sexo','uf','cep'),
    estimate_without_term_frequencies=True,
)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primeiro_nome_phon" = r."primeiro_nome_phon") AND (l."ultimo_nome_phon" = r."ultimo_nome_phon") AND (l."sexo" = r."sexo") AND (l."uf" = r."uf") AND (l."cep" = r."cep")

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - nome_meio_phon
    - data_nascimento
    - idade

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - primeiro_nome_phon
    - ultimo_nome_phon
    - sexo
    - uf


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Iteration 1: Largest change in params was -0.904 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 2: Largest change in params was -0.264 in the m_probability of idade, level `Exact match on idade`
Iteration 3: Largest change in params was -0.178 in the m_probability of nome_completo_phon, level `Exact match on nome_completo_phon`
Iteration 4: Largest change in params was 0.337 in the m_probability of nome_meio_phon, level `All other comparisons`
Iteration 5: Largest change in params was 0.0545 in the m_probability of nome_meio_phon, level `All other comparisons`
Iteration 6: Largest change in params was 0.0018 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 7: Largest change in params was 0.000849 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 8: Largest change in params was 0.00039 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 9: Largest ch

<EMTrainingSession, blocking on (l."primeiro_nome_phon" = r."primeiro_nome_phon") AND (l."ultimo_nome_phon" = r."ultimo_nome_phon") AND (l."sexo" = r."sexo") AND (l."uf" = r."uf") AND (l."cep" = r."cep"), deactivating comparisons primeiro_nome_phon, ultimo_nome_phon, sexo, uf>

In [33]:
from config import MODELS_DIR

SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
linker.misc.save_model_to_json(str(MODELS_DIR / 'splink_model.json'), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)
print('Cópia em models/:', MODELS_DIR / 'splink_model.json')


Modelo salvo: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json
Cópia em models/: /home/ibge.gov.br/ramon.goncalves/ibge-probabilistico/models/splink_model.json


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).

In [34]:
linker.visualisations.match_weights_chart()


/opt/venvs/singed/jhub/lib64/python3.9/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [35]:
linker.evaluation.unlinkables_chart()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.LayerChart(...)

## Predict + clustering

`predict(0.5)` só gera candidatos. Clustering e validação usam **0,95**.


In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Blocking time: 476.58 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time (post-blocking): 727.75 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'uf':
    m values not fully trained
Comparison: 'uf':
    u values not fully trained


In [ ]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.98,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())

## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).

In [ ]:
records_to_plot = df_predictions.tail(5).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)


#records_sample = df_predictions.to_dict(orient='records')
#Linker.visualisations.waterfall_chart(records_sample, filter_nulls=False)

In [ ]:
pd.set_option('display.max_columns', None)
#df_predictions.head(5)
#df_predictions['unique_id_l'].str.contains("censo").head(5)
df_predictions[(df_predictions['unique_id_l'].str.contains("censo")) & df_predictions['unique_id_r'].str.contains("cpf")].head(5)




## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).

In [ ]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))

## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Labels Splink no NB03;
funil da lista ouro no NB04. Cheque os match weights: discordar nome/DOB
deve penalizar vários bits, não ≈ 0.


In [ ]:
display(df_predictions['match_probability'].describe())
faixas = pd.cut(df_predictions['match_weight'], bins=20)
display(
    df_predictions.groupby(faixas, observed=True)
    .size()
    .rename('n_pares')
    .to_frame()
)

In [ ]:
tamanhos = df_clusters.groupby('cluster_id').size()
print(f'Clusters: {tamanhos.size:,} | maior: {tamanhos.max():,} | singletons: {(tamanhos == 1).sum():,}')
display(
    tamanhos.value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)

# Clusters grandes demais indicam blocking/threshold frouxo — inspecionar antes do NB03.
display(tamanhos.sort_values(ascending=False).head(10).rename('tamanho').to_frame())

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(SPLINK_PREDICTIONS)
df_clusters.to_parquet(SPLINK_CLUSTERS)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)

## Encerrar

Artefatos prontos para o [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb).

In [ ]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')

In [ ]:
con.close()